# Doppler effect and Mach cone

*waves · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/waves/doppler_effect)

A source moving at speed $v_s$ emits one circular wavefront per period. Every wavefront is a row of a tensor (emission time, emission point), so its radius at any time is one subtraction. Fronts bunch up ahead and spread out behind; above the speed of sound they pile up into a Mach cone.

**Vector form**

$$r_e(t) = c\,(t - t_e),\qquad f' = \frac{f\,c}{c \mp v_s},\qquad \sin\mu = \frac{c}{v_s}$$

**Index (tensor) form**

$$t_{oe} = t_e + \tfrac{1}{c}\sqrt{(o_{oi} - s_{ei})(o_{oi} - s_{ei})},\qquad f'_o = \frac{E - 1}{t_{o,\text{last}} - t_{o,\text{first}}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $t_e$ | `emission_times(f, t_max, dev)` | (E,) | one wavefront every period $1/f$ |
| $s_{ei}$ | `source_position(t_e, v_s, x_start)` | (E, 2) | where the source was when it emitted front $e$ |
| $r_e(t)$ | `front_radii(t, t_e, c)` | (E,) | radius of every front at time $t$ (zero if not yet emitted) |
| $o_{oi}$ | `observers` | (O, 2) | listeners ahead of and behind the source |
| $(o_{oi} - s_{ei})(o_{oi} - s_{ei})$ | `einsum('oei,oei->oe', d, d)` | (O, E) | repeated $i$ ⇒ summed: squared distance from each emission point to each listener |
| $t_{oe}$ | `arrival_times(observers, s, t_e, c)` | (O, E) | when front $e$ reaches listener $o$ |

The heard frequency is simply the number of fronts per second that arrive: ahead of the source the fronts were emitted closer together, so $f' = f\,c/(c - v_s)$; behind, $f' = f\,c/(c + v_s)$. Listeners are placed on the line of motion where these formulas are exact. For $v_s > c$ (Mach number $M = v_s/c > 1$) the source outruns its own sound: all fronts touch two straight lines, the Mach cone of half-angle $\mu$ with $\sin\mu = 1/M$. The listener ahead then hears nothing until the cone passes, and the fronts arrive in reverse order. Speed of sound $c = 340$ m/s; only the source moves, the air is still.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/waves/doppler_effect/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    c, f, t_max = tensors.C_SOUND, params["f"], params["t_max"]
    v_s = params["mach"] * c
    x_start = -0.5 * v_s * t_max  # keep the source's path centred on the origin

    t_e = tensors.emission_times(f, t_max, dev)  # (E,)
    s = tensors.source_position(t_e, v_s, x_start)  # (E, 2)
    obs = tensors.observers(x_start, v_s, t_max, c, dev)  # (2, 2)
    result = solve.solve(t_e, s, obs, c, t_max)

    heard = equation.heard_frequency(result["arrivals"]).tolist()
    ahead, behind = equation.doppler_formula(f, c, v_s)
    print(f"source speed v_s = {v_s:.0f} m/s  (Mach {params['mach']:.2f}),  emitted f = {f:.2f} Hz")
    print(f"ahead : heard {heard[0]:.3f} Hz   formula f c/|c - v_s| = {ahead:.3f} Hz")
    print(f"behind: heard {heard[1]:.3f} Hz   formula f c/(c + v_s)  = {behind:.3f} Hz")
    if params["mach"] > 1:
        mu = equation.cone_half_angle(
            s, result["radii"][-1], tensors.source_position(t_e.new_tensor(t_max), v_s, x_start)
        )
        exact = math.degrees(math.asin(c / v_s))
        print(f"Mach cone half-angle {math.degrees(mu.mean().item()):.2f}°  (asin(c/v_s) = {exact:.2f}°)")
        print("ahead of a supersonic source the fronts arrive in reverse order, all after the cone passes")
    return plot.animate(result, t_e, s, obs, v_s, c, out_dir / "doppler_effect.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/waves/doppler_effect/equation.py
r"""Physics → tensor translation.

A point source moving along x emits one wavefront at each time t_e (e = 0 … E−1) from the point s_{ei}.
In still air every front is a circle that grows at the speed of sound:

    r_e(t) = c (t - t_e)        for t ≥ t_e

Front e reaches listener o (position o_{oi}) at

    t_{oe} = t_e + \frac{1}{c}\sqrt{(o_{oi} - s_{ei})(o_{oi} - s_{ei})}        (i summed)

and the listener hears as many fronts per second as arrive: f'_o = (E − 1) / (t_{o,last} − t_{o,first}).
On the line of motion this gives f' = f c / (c ∓ v_s). For v_s > c the fronts touch a cone with sin μ = c / v_s.
"""

import math

import torch


def front_radii(t: float, t_e: torch.Tensor, c: float) -> torch.Tensor:
    """r_e = c (t − t_e), clipped at 0 for fronts not emitted yet.   t_e: (E,) → (E,)"""
    return torch.clamp(c * (t - t_e), min=0.0)


def arrival_times(observers: torch.Tensor, s: torch.Tensor, t_e: torch.Tensor, c: float) -> torch.Tensor:
    r"""t_{oe} = t_e + |o_o − s_e| / c.   observers: (O, 2), s: (E, 2), t_e: (E,) → (O, E)"""
    d = observers[:, None, :] - s[None, :, :]  # (O, E, 2)
    dist = torch.sqrt(torch.einsum("oei,oei->oe", d, d))  # i summed
    return t_e[None, :] + dist / c


def heard_frequency(arrivals: torch.Tensor) -> torch.Tensor:
    """f'_o = (number of fronts − 1) / (time between first and last arrival).   (O, E) → (O,)"""
    n = arrivals.shape[1]
    span = arrivals.max(dim=1).values - arrivals.min(dim=1).values
    return (n - 1) / span


def doppler_formula(f: float, c: float, v_s: float) -> tuple[float, float]:
    """Textbook (ahead, behind) frequencies f c / (c − v_s) and f c / (c + v_s); |…| once v_s > c."""
    ahead = math.inf if abs(c - v_s) < 1e-9 else f * c / abs(c - v_s)
    return ahead, f * c / (c + v_s)


def cone_half_angle(s: torch.Tensor, radii: torch.Tensor, source_now: torch.Tensor) -> torch.Tensor:
    """Half-angle of the lines from the source that touch the fronts: sin μ_e = r_e / |source − s_e|.

    s: (E, 2), radii: (E,), source_now: (2,). For v_s > c every front gives the same μ: the Mach cone.
    """
    d = source_now[None, :] - s  # (E, 2)
    dist = torch.sqrt(torch.einsum("ei,ei->e", d, d))
    ok = (radii > 0) & (dist > radii)  # the source is outside the front
    return torch.asin(radii[ok] / dist[ok])


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/waves/doppler_effect/tensors.py
"""Build the concrete tensors: emission times, emission points and the two listeners."""

import torch

C_SOUND = 340.0  # m/s


def emission_times(f: float, t_max: float, device: torch.device) -> torch.Tensor:
    """t_e = 0, 1/f, 2/f, … < t_max: (E,)."""
    return torch.arange(0.0, t_max, 1.0 / f, device=device)


def source_position(t: torch.Tensor, v_s: float, x_start: float) -> torch.Tensor:
    """s_i(t) = (x_start + v_s t, 0) for every time in t: (…, 2)."""
    x = x_start + v_s * t
    return torch.stack([x, torch.zeros_like(x)], dim=-1)


def observers(x_start: float, v_s: float, t_max: float, c: float, device: torch.device) -> torch.Tensor:
    """o_{oi} of shape (2, 2): one listener ahead of where the source ends, one behind where it starts."""
    margin = 0.15 * c * t_max
    return torch.tensor([[x_start + v_s * t_max + margin, 0.0], [x_start - margin, 0.0]], device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/waves/doppler_effect/solve.py
"""No time stepping: the radius of every front is known at every time, so we just tabulate it."""

import torch

from .equation import arrival_times, front_radii


def solve(t_e: torch.Tensor, s: torch.Tensor, obs: torch.Tensor, c: float, t_max: float, n_times: int = 200) -> dict:
    """Return times (T,), radii (T, E) of every front at every time, and arrival times (O, E) at the listeners."""
    times = torch.linspace(0.0, t_max, n_times, device=t_e.device)
    radii = torch.stack([front_radii(t, t_e, c) for t in times.tolist()])  # (T, E)
    return {"times": times, "radii": radii, "arrivals": arrival_times(obs, s, t_e, c)}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/waves/doppler_effect/plot.py
"""Animate the growing wavefronts, the moving source, the Mach cone (if any) and what each listener hears."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from matplotlib.patches import Circle

from problems._common.plotting import frame_indices, save_gif


def animate(result: dict, t_e: torch.Tensor, s: torch.Tensor, obs: torch.Tensor, v_s: float, c: float, path: Path):
    times = result["times"].cpu().numpy()  # (T,)
    radii = result["radii"].cpu().numpy()  # (T, E)
    arrivals = result["arrivals"].cpu().numpy()  # (O, E)
    s_np, obs_np, te = s.cpu().numpy(), obs.cpu().numpy(), t_e.cpu().numpy()
    x_start = s_np[0, 0]

    fig, ax = plt.subplots(figsize=(7, 4))
    x_lo, x_hi = obs_np[1, 0] - 0.05 * c * times[-1], obs_np[0, 0] + 0.05 * c * times[-1]
    half_height = 0.45 * c * times[-1]  # tall enough to show most fronts
    x_mid, half_width = 0.5 * (x_lo + x_hi), max(0.5 * (x_hi - x_lo), 1.75 * half_height)
    ax.set_xlim((x_mid - half_width) / 1000, (x_mid + half_width) / 1000)
    ax.set_ylim(-half_height / 1000, half_height / 1000)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")

    circles = [Circle((sx / 1000, 0), 0, fill=False, ec="#2563eb", lw=1.2) for sx in s_np[:, 0]]
    for circle in circles:
        ax.add_patch(circle)
    source = ax.plot([], [], "o", color="#1e40af", ms=8, zorder=5)[0]
    cone = [ax.plot([], [], color="#f59e0b", lw=2)[0] for _ in range(2)]
    labels = []
    for o in obs_np:
        ax.plot([o[0] / 1000], [0], "^", color="#dc2626", ms=9, zorder=5)
        labels.append(
            ax.text(o[0] / 1000, 0.08 * half_height / 1000, "", ha="center", va="bottom", fontsize=8,
                    bbox={"fc": "white", "ec": "none", "alpha": 0.8, "pad": 1})
        )  # fmt: skip
    title = ax.set_title("M = 0.00   source f = 0.00 Hz   t = 0.0 s")  # placeholder for tight_layout
    mach = v_s / c

    def update(step: int):
        t = times[step]
        for circle, r in zip(circles, radii[step]):
            circle.set_radius(r / 1000)
            circle.set_visible(r > 0)
        xs = x_start + v_s * t
        source.set_data([xs / 1000], [0])
        # Mach cone: two lines from the source, touching all fronts, half-angle μ with sin μ = c / v_s
        if mach > 1 and t > 0:
            mu = math.asin(1 / mach)
            length = v_s * t * math.cos(mu)  # distance from the source to where the cone touches the oldest front
            for sign, line in zip((1, -1), cone):
                line.set_data(
                    [xs / 1000, (xs - length * math.cos(mu)) / 1000], [0, sign * length * math.sin(mu) / 1000]
                )
        for label, name, arr in zip(labels, ["ahead", "behind"], arrivals):
            heard = arr[arr <= t]
            if len(heard) >= 2:
                label.set_text(f"{name}\nf' = {(len(heard) - 1) / (heard.max() - heard.min()):.2f} Hz")
            else:
                label.set_text(f"{name}\n(silent)")
        title.set_text(f"M = {mach:.2f}   source f = {1 / (te[1] - te[0]):.2f} Hz   t = {t:.1f} s")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(times), 90), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'mach': 0.6,
    'f': 1.0,
    't_max': 10.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.waves.doppler_effect")]:
    del sys.modules[name]
main = importlib.import_module("problems.waves.doppler_effect.main")
problem_dir = Path("problems/waves/doppler_effect")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
